# Cortex APT Multi-Agent RL\n\nNotebook Google Colab complet pour Cortex.\n\nCe pipeline produit uniquement des **risk signals** pour Cortex. Il ne prend jamais la décision finale.

## 1. Setup

In [ ]:
!pip -q install pandas numpy networkx scikit-learn matplotlib seaborn tqdm\n

In [ ]:
import sys\nfrom pathlib import Path\nimport torch\n\nROOT = Path('/content/coco')\nif ROOT.exists() and str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\n\nfrom scripts.runtime.cortex_apt_multiagent_rl_module import (\n    ACTIONS,\n    NUMERIC_COLS,\n    build_verified_colab_payload,\n    compute_dataset_fingerprint,\n    evaluate_runtime,\n    export_results,\n    fit_anomaly_model,\n    push_verified_colab_payload,\n    run_pipeline,\n    save_verified_colab_payload,\n    set_seed,\n    simulate_dataset,\n    train_gcn,\n    train_sentinel_dqn,\n    build_graph_tensors,\n    visualize_results,\n)\n\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nset_seed(42)\nprint('device:', device)\n

## 2. Dataset Simulation (APT)

In [ ]:
graph, df = simulate_dataset(seed=42)\nprint('nodes:', graph.number_of_nodes(), 'edges:', graph.number_of_edges())\nprint('events:', len(df), 'attacks:', int(df['label_attack'].sum()), 'zero_day:', int(df['label_zero_day'].sum()))\ndf.head()\n

## 3. Feature Engineering

In [ ]:
df[NUMERIC_COLS + ['label_attack', 'label_zero_day']].describe().T\n

## 4. Graph Construction

In [ ]:
node_index, node_features, adjacency, node_labels = build_graph_tensors(graph, df)\ngcn, node_scores, node_embeddings, gcn_loss = train_gcn(node_features, adjacency, node_labels, device=device)\nfloat(node_scores.mean()), float(node_scores.max())\n

## 5. Multi-Agent System\n\nLe module construit implicitement les 5 agents Cortex:\n- Sentinel Agent\n- Threat Hunter Agent\n- Trust Evaluator Agent\n- Graph Analyst Agent\n- Anomaly Detector Agent

## 6. Campaign Memory Engine\n\nLa mémoire de campagne est intégrée dans le pipeline runtime pour accumuler les signaux faibles multi-phase.

## 7. Hybrid Model

In [ ]:
from sklearn.model_selection import train_test_split\n\ntrain_df, test_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['label_attack'])\ndetector = fit_anomaly_model(train_df)\nprint('train:', len(train_df), 'test:', len(test_df))\n

## 8. Reinforcement Learning (Sentinel)

In [ ]:
q_net, rl_rewards = train_sentinel_dqn(train_df, detector, node_index, node_scores, device=device)\nrl_rewards[-5:]\n

## 9. Communication Inter-Agents\n\nLe bus de messages est utilisé pendant `run_pipeline`, avec broadcast contrôlé et historique exploitable.

## 10. Filtering & Priority\n\nLe moteur de priorité utilise la formule Cortex fournie pour filtrer `filtered / review_queue / dropped`.

## 11. Agent Routing\n\nLe routage produit: `Sentinel`, `Threat Hunter`, `Trust Engine`, `Incident Response`, `SOC`.

## 12. Real-time APT Simulation

In [ ]:
runtime_df, agent_logs = run_pipeline(test_df, detector, node_index, node_scores, q_net, device=device)\nruntime_df[['scenario', 'phase', 'sentinel_score', 'priority', 'filter_bucket', 'route', 'rl_action']].head(12)\n

## 13. Evaluation

In [ ]:
metrics = evaluate_runtime(runtime_df)\nmetrics['attack_metrics'], metrics['zero_day_metrics'], metrics['low_and_slow_detection'], metrics['false_positive_rate']\n

In [ ]:
metrics['agent_performance']\n

## 14. Visualization

In [ ]:
visualize_results(graph, runtime_df, rl_rewards)\n

## 15. Export

In [ ]:
export_manifest = export_results('/content/cortex_exports', gcn, q_net, runtime_df, agent_logs, metrics)\nexport_manifest\n

## 16. Cortex Verified Sync\n\nCette section construit un payload verifie compatible avec `cortex-orchestrator` et permet un push signe vers Cortex.

In [ ]:
RUN_ID = 'colab-run-apt-001'\nTRAINING_PLAN_ID = 'plan-apt-multiagent-001'\nTARGET_AGENTS = ['decision', 'sentinel', 'threat_hunter']\nKNOWLEDGE_REGISTRY_FINGERPRINT = 'known-registry-update-me'\nREVIEWER = 'analyst@cortex.local'\nNOTES = 'Verified Colab APT simulation. Risk signals only, no execution authority granted.'\n\nverified_payload = build_verified_colab_payload(\n    runtime_df,\n    metrics,\n    run_id=RUN_ID,\n    training_plan_id=TRAINING_PLAN_ID,\n    target_agents=TARGET_AGENTS,\n    knowledge_registry_fingerprint=KNOWLEDGE_REGISTRY_FINGERPRINT,\n    reviewer=REVIEWER,\n    notes=NOTES,\n)\nverified_payload_path = save_verified_colab_payload(verified_payload, '/content/cortex_exports')\nverified_payload_path, verified_payload['dataset_fingerprint'], len(verified_payload['accepted_item_ids'])\n

In [ ]:
# Renseigner ces variables pour pousser le resultat vers Cortex.\nCORTEX_SYNC_URL = 'http://127.0.0.1:18082/v1/training/colab/ingest'\nCORTEX_SHARED_SECRET = ''\nENABLE_PUSH = False\n\npush_result = {'status': 'disabled'}\nif ENABLE_PUSH:\n    if not CORTEX_SHARED_SECRET:\n        raise ValueError('CORTEX_SHARED_SECRET must be set before ENABLE_PUSH=True')\n    push_result = push_verified_colab_payload(\n        verified_payload,\n        url=CORTEX_SYNC_URL,\n        secret=CORTEX_SHARED_SECRET,\n    )\npush_result\n

## Cortex Pipeline Note\n\nLes sorties exportées servent à alimenter Cortex en **risk signals** et en résultats d'entraînement vérifiés.\nLes décisions finales restent bornées par policy, trust, approval et modes dégradés côté Cortex.